In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/OE/Kerendia_OE_2.0/00_Helper_Notebooks/00_connection"

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/OE/Kerendia_OE_2.0/00_Helper_Notebooks/00_config"

In [0]:
# df_nba_all = spark.read.format("delta").load(
#     "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/oe_journey/oe_volume_space/notebook_output/oe_final_nba_all_features_v1"
# )

df_nba_all = spark.read.format("delta").table("`ph-com-ai-us-prod-catalog-2553213839599380`.kerendia_oe_journey.oe_final_nba_all_features_v2")

In [0]:
df_nba_all.count()

In [0]:
df_nba_all.display()

In [0]:
len(df_nba_all.columns)

In [0]:
df_nba_all.columns

In [0]:
split_col = F.split(F.col("HCP_COHORT_ID"), "_")

# F.get() returns null if index doesn't exist — safe for malformed/missing values
df_nba_all = df_nba_all.withColumn("PROVIDER_ID", F.get(split_col, 0))
df_nba_all = df_nba_all.withColumn("COHORT",      F.get(split_col, 1))

In [0]:
df_nba_all=df_nba_all.drop("HCP_COHORT_ID") 

In [0]:
df_nba_all_01= df_nba_all.withColumn(
    "instance",
    F.when(F.col("COHORT") == "C03", 1)
     .when(F.col("COHORT") == "C06", 2)
     .when(F.col("COHORT") == "C09", 3)
     .when(F.col("COHORT") == "C12", 4)
     .when(F.col("COHORT") == "C15", 5)
     .when(F.col("COHORT") == "C18", 6)
     .when(F.col("COHORT") == "C21", 7)
)

In [0]:
df_nba_all_01 = df_nba_all_01.filter(F.col("COHORT").isNotNull())

In [0]:
df_nba_all_01 = df_nba_all_01.filter(F.col("instance").isNotNull())

In [0]:
df_nba_all_01.select("instance").distinct().display()

In [0]:
df_nba_all_02 = df_nba_all_01.filter(F.col("instance").isin([1,2,3,4,5,6,7]))

In [0]:
df_nba_all_02.select("COHORT").distinct().display()

In [0]:
df_nba_all_02 = df_nba_all_02.drop("COHORT")

In [0]:
df_nba_all_03 = df_nba_all_02.filter(col("INSTANCE").isin([7]))


In [0]:
from math import ceil
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType, DoubleType

# ----------------------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------------------
OUTPUT_PATH = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_oe_journey/kerendia_oe_volume/notebook_output/oe_eda"

DROP_COLS = ["PROVIDER_ID", "INSTANCE"]

CHUNK_SIZE = 150   # safer for ultra-wide DF

# ----------------------------------------------------------------------------
# 1. PREP DF
# ----------------------------------------------------------------------------

cols_to_keep = [c for c in df_nba_all_03.columns if c not in DROP_COLS]

# single projection instead of repeated withColumn
select_exprs = []

for f in df_nba_all_03.schema.fields:

    if f.name in DROP_COLS:
        continue

    if isinstance(f.dataType, DecimalType):
        select_exprs.append(
            F.col(f.name).cast(DoubleType()).alias(f.name)
        )
    else:
        select_exprs.append(F.col(f.name))

sdf = df_nba_all_03.select(*select_exprs)

# ----------------------------------------------------------------------------
# 2. ROW COUNT
# ----------------------------------------------------------------------------

n_rows = sdf.count()

all_cols = sdf.columns
n_cols = len(all_cols)

print(f"Rows: {n_rows:,}")
print(f"Columns: {n_cols:,}")

# ----------------------------------------------------------------------------
# 3. CHUNKED AGG
# ----------------------------------------------------------------------------

summary_rows = []

n_chunks = ceil(n_cols / CHUNK_SIZE)

for i in range(n_chunks):

    chunk_cols = all_cols[
        i * CHUNK_SIZE : (i + 1) * CHUNK_SIZE
    ]

    agg_exprs = [
        F.count(F.col(c)).alias(c)
        for c in chunk_cols
    ]

    result = (
        sdf
        .agg(*agg_exprs)
        .first()
        .asDict()
    )

    for c in chunk_cols:

        non_null = result.get(c, 0)

        fill_rate = (
            non_null / n_rows
            if n_rows > 0 else None
        )

        summary_rows.append(
            (c, n_rows, fill_rate)
        )

    print(f"Chunk {i+1}/{n_chunks} done")

# ----------------------------------------------------------------------------
# 4. OUTPUT DF
# ----------------------------------------------------------------------------

out_sdf = spark.createDataFrame(
    summary_rows,
    ["feature", "n", "fill_rate"]
)

display(out_sdf)

# ----------------------------------------------------------------------------
# 5. SAVE
# ----------------------------------------------------------------------------

(
    out_sdf.write
    .mode("overwrite")
    .format("delta")
    .option("overwriteSchema", "true")
    .save(OUTPUT_PATH)
)

print("Done")

In [0]:
mdb_df = spark.read.format("delta").load("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/oe_journey/oe_volume_space/notebook_output/model_training_mdb_nba_features_v2")

In [0]:
mdb_df.columns

In [0]:
mdb_df01 = mdb_df.withColumnRenamed("CUSTOMER_ID","BH_ID")

In [0]:
npi_bhid=spark.read.format("delta").load(
    "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/oe_journey/oe_volume_space/query_output/kerendia_nba_3_0_npi_bhid_mapping"
)


In [0]:
npi_bhid = npi_bhid.select("BH_ID","PROVIDER_ID").distinct()

In [0]:
mdb_df02 = mdb_df01.join(npi_bhid, on="BH_ID", how="inner")

In [0]:
mdb_df02 = mdb_df02.drop("BH_ID")

In [0]:
mdb_df02.display()

In [0]:
 provider_df01 = mdb_df02.select("PROVIDER_ID").distinct()

In [0]:
inst_df = mdb_df02.filter((col("instance") == 6) | (col("instance") == 7)).select("instance").distinct()

In [0]:
skeleton_df = provider_df01.crossJoin(inst_df)

In [0]:
skeleton_df = skeleton_df.withColumnRenamed("instance", "INSTANCE")

In [0]:
final_df01 = skeleton_df.join(
    df_nba_all_02,
    on=["PROVIDER_ID", "INSTANCE"],
    how="left"
)

In [0]:
final_df01.display()

In [0]:
cols_to_keep = [c for c in mdb_df02.columns if c.endswith("P3")] + ["TOTAL_NBRX_C3"] + ["PROVIDER_ID", "INSTANCE"]

mdb_df03 = mdb_df02.select(cols_to_keep)

In [0]:
mdb_df03.display()

In [0]:
final_df02 = skeleton_df.join(
    mdb_df03,
    on=["PROVIDER_ID", "INSTANCE"],
    how="left"
)

In [0]:
final_df02.display()

In [0]:
final_mdb  = final_df01.join(final_df02, on=["PROVIDER_ID", "INSTANCE"], how="left")

In [0]:
final_mdb.display()

In [0]:
# /Workspace/Users/wenjie.chen@bayer.com/OE/OE_2.0/Feature Selection/01_create_mdb --path of the mdb 

In [0]:
final_mdb.write.mode('overwrite') \
    .format("delta") \
    .save("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/oe_journey/oe_volume_space/notebook_output/final_mdb")